# 🗃️ SQL Analysis / SQL-Analyse — E-Commerce Sales
**Author / Autor:** Yurii Oleschuk  
**Engine:** SQLite via Python (sqlite3 + pandas)  

### 🇬🇧 Note
All queries run on a cleaned view that excludes returns, invalid prices, and non-product entries.

### 🇩🇪 Hinweis
Alle Abfragen basieren auf einer bereinigten Ansicht, die Rücksendungen, ungültige Preise und Nicht-Produkteinträge ausschließt.

## Setup — Load Data into SQLite / Daten in SQLite laden

In [ ]:
# EN Import libraries
# DE Bibliotheken importieren
import pandas as pd
import sqlite3
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from IPython.display import display
import warnings
warnings.filterwarnings('ignore')

pd.options.display.float_format = '{:,.2f}'.format
sns.set_theme(style='whitegrid')
plt.rcParams.update({'figure.dpi': 110, 'font.size': 11})

# EN Load CSV into in-memory SQLite database
# DE CSV in SQLite-Datenbank im Arbeitsspeicher laden
df_raw = pd.read_csv('../data/sales.csv', encoding='ISO-8859-1')
conn   = sqlite3.connect(':memory:')
df_raw.to_sql('sales', conn, if_exists='replace', index=False)

# EN Create cleaned view (base for all queries below)
# DE Bereinigte Ansicht erstellen (Grundlage für alle Abfragen)
conn.execute("""
CREATE VIEW clean_sales AS
SELECT *, Quantity * UnitPrice AS Revenue
FROM   sales
WHERE  Quantity  > 0
  AND  UnitPrice > 0
  AND  CustomerID IS NOT NULL
  AND  Description NOT LIKE '%POSTAGE%'
  AND  Description NOT LIKE '%DOTCOM%'
  AND  Description NOT LIKE '%MANUAL%'
  AND  Description NOT LIKE '%BANK CHARGES%'
""")

raw_count   = df_raw.shape[0]
clean_count = pd.read_sql('SELECT COUNT(*) AS n FROM clean_sales', conn)['n'][0]
print('✅ Data loaded into SQLite / Daten in SQLite geladen')
print(f'   Raw rows / Rohdaten:    {raw_count:,}')
print(f'   Clean rows / Bereinigt: {clean_count:,} ({clean_count/raw_count:.1%} retained)')

## Query 1 — KPI Summary / Kennzahlenübersicht

In [ ]:
# EN Single-row KPI snapshot — ideal for a dashboard header
# DE Einzeilige KPI-Übersicht — ideal für Dashboard-Header
kpi = pd.read_sql("""
SELECT
    ROUND(SUM(Revenue), 2)                               AS total_revenue,
    COUNT(DISTINCT InvoiceNo)                            AS total_orders,
    COUNT(DISTINCT CustomerID)                           AS unique_customers,
    ROUND(SUM(Revenue) / COUNT(DISTINCT InvoiceNo), 2)  AS avg_order_value,
    ROUND(SUM(Revenue) / COUNT(DISTINCT CustomerID), 2) AS avg_revenue_per_customer
FROM clean_sales
""", conn)

display(kpi)

print('\n🇬🇧 Insight: These 5 KPIs form the baseline for any business health report.')
print('   Track them monthly to detect growth or decline early.')
print('\n🇩🇪 Erkenntnis: Diese 5 Kennzahlen bilden die Grundlage jedes Unternehmensberichts.')
print('   Monatliches Monitoring ermöglicht frühzeitiges Erkennen von Wachstum oder Rückgang.')

## Query 2 — Top 10 Products by Revenue / Top-10-Produkte nach Umsatz

In [ ]:
# EN Top products with cumulative Pareto percentage
# DE Top-Produkte mit kumulativem Pareto-Anteil
top_products = pd.read_sql("""
WITH product_revenue AS (
    SELECT
        Description,
        ROUND(SUM(Revenue), 2)    AS revenue,
        SUM(Quantity)             AS total_qty,
        COUNT(DISTINCT InvoiceNo) AS order_count
    FROM   clean_sales
    GROUP  BY Description
),
total AS (SELECT SUM(revenue) AS grand_total FROM product_revenue)
SELECT
    p.Description,
    p.revenue,
    p.total_qty,
    p.order_count,
    ROUND(p.revenue * 100.0 / t.grand_total, 2) AS revenue_pct
FROM   product_revenue p, total t
ORDER  BY p.revenue DESC
LIMIT  10
""", conn)

display(top_products)

fig, ax = plt.subplots(figsize=(12, 5))
colors = sns.color_palette('Blues_r', n_colors=10)
ax.barh(top_products['Description'][::-1], top_products['revenue'][::-1], color=colors)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'£{x/1000:.0f}k'))
ax.set_title('Top 10 Products by Revenue / Top-10-Produkte nach Umsatz', fontweight='bold')
ax.set_xlabel('Revenue / Umsatz (£)')
plt.tight_layout()
plt.show()

print('\n🇬🇧 Insight: A small number of products drive the majority of revenue (Pareto principle).')
print('   Focus inventory investment and marketing on these core SKUs.')
print('\n🇩🇪 Erkenntnis: Wenige Produkte generieren den Großteil des Umsatzes (Pareto-Prinzip).')
print('   Lager und Marketing auf diese Kernprodukte konzentrieren.')

## Query 3 — Monthly Revenue Trend with MoM Growth / Monatlicher Umsatztrend mit MoM-Wachstum

In [ ]:
# EN Monthly aggregation with growth calculated in Python
# DE Monatliche Aggregation, Wachstum in Python berechnet
monthly = pd.read_sql("""
SELECT
    STRFTIME('%Y-%m', InvoiceDate) AS month,
    ROUND(SUM(Revenue), 2)         AS revenue,
    COUNT(DISTINCT InvoiceNo)      AS orders,
    COUNT(DISTINCT CustomerID)     AS customers
FROM   clean_sales
GROUP  BY month
ORDER  BY month
""", conn)

# EN Month-over-month growth / DE Monatliches Wachstum
monthly['mom_growth_pct'] = monthly['revenue'].pct_change() * 100

display(monthly)

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

axes[0].bar(monthly['month'], monthly['revenue'], color='steelblue', alpha=0.85)
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'£{x/1000:.0f}k'))
axes[0].set_title('Monthly Revenue / Monatlicher Umsatz', fontweight='bold')
axes[0].set_ylabel('Revenue / Umsatz (£)')

colors_mom = ['green' if x >= 0 else 'red' for x in monthly['mom_growth_pct'].fillna(0)]
axes[1].bar(monthly['month'], monthly['mom_growth_pct'].fillna(0), color=colors_mom, alpha=0.8)
axes[1].axhline(0, color='black', linewidth=0.8)
axes[1].set_title('Month-over-Month Growth (%) / Monat-für-Monat-Wachstum (%)', fontweight='bold')
axes[1].set_ylabel('MoM %')

for ax in axes:
    ax.tick_params(axis='x', rotation=45)
    ax.grid(axis='y', alpha=0.4)

plt.tight_layout()
plt.show()

peak = monthly.loc[monthly['revenue'].idxmax()]
print(f'\n🇬🇧 Insight: Peak month is {peak["month"]} (£{peak["revenue"]:,.0f}). Strong Q4 seasonality.')
print('   Negative MoM after a peak signals end of seasonal demand — reduce stock exposure.')
print(f'\n🇩🇪 Erkenntnis: Stärkster Monat ist {peak["month"]} (£{peak["revenue"]:,.0f}). Starke Q4-Saisonalität.')
print('   Negatives MoM nach dem Höhepunkt signalisiert das Ende der Saison — Lagerrisiko reduzieren.')

## Query 4 — Average Order Value / Durchschnittlicher Bestellwert

In [ ]:
# EN Overall AOV / DE Gesamt-Ø-Bestellwert
overall_aov = pd.read_sql("""
SELECT ROUND(AVG(order_value), 2) AS overall_aov
FROM (
    SELECT InvoiceNo, SUM(Revenue) AS order_value
    FROM   clean_sales
    GROUP  BY InvoiceNo
)
""", conn)

# EN AOV by country (excl. UK) / DE Ø-Bestellwert nach Land (ohne UK)
aov_by_country = pd.read_sql("""
WITH order_vals AS (
    SELECT InvoiceNo, Country, SUM(Revenue) AS order_value
    FROM   clean_sales
    GROUP  BY InvoiceNo, Country
)
SELECT
    Country,
    COUNT(*)                    AS orders,
    ROUND(AVG(order_value), 2)  AS aov,
    ROUND(SUM(order_value), 2)  AS total_revenue
FROM   order_vals
WHERE  Country != 'United Kingdom'
GROUP  BY Country
ORDER  BY aov DESC
LIMIT  10
""", conn)

print('Overall AOV / Gesamt-Ø-Bestellwert:')
display(overall_aov)
print('\nAOV by Country (excl. UK) / Ø-Bestellwert nach Land (ohne UK):')
display(aov_by_country)

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(aov_by_country['Country'][::-1], aov_by_country['aov'][::-1],
        color='teal', alpha=0.8)
ax.set_title('AOV by Country (excl. UK) / Ø-Bestellwert nach Land (ohne UK)', fontweight='bold')
ax.set_xlabel('Avg Order Value / Ø Bestellwert (£)')
plt.tight_layout()
plt.show()

print('\n🇬🇧 Insight: High-AOV countries likely have B2B wholesale buyers.')
print('   Target them with volume discounts and direct sales outreach.')
print('\n🇩🇪 Erkenntnis: Länder mit hohem Ø-Bestellwert haben wahrscheinlich B2B-Großhandelskäufer.')
print('   Mit Mengenrabatten und direkter Vertriebsansprache ansprechen.')

## Query 5 — Top 10 Customers / Top-10-Kunden

In [ ]:
# EN Top customers by total revenue with order history
# DE Top-Kunden nach Gesamtumsatz mit Bestellhistorie
top_customers = pd.read_sql("""
SELECT
    CustomerID,
    ROUND(SUM(Revenue), 2)                              AS total_spent,
    COUNT(DISTINCT InvoiceNo)                           AS orders_count,
    ROUND(SUM(Revenue) / COUNT(DISTINCT InvoiceNo), 2) AS customer_aov,
    MIN(DATE(InvoiceDate))                              AS first_purchase,
    MAX(DATE(InvoiceDate))                              AS last_purchase
FROM   clean_sales
GROUP  BY CustomerID
ORDER  BY total_spent DESC
LIMIT  10
""", conn)

display(top_customers)

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(top_customers['CustomerID'].astype(str), top_customers['total_spent'],
       color=sns.color_palette('Blues_r', 10))
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'£{x/1000:.0f}k'))
ax.set_title('Top 10 Customers by Revenue / Top-10-Kunden nach Umsatz', fontweight='bold')
ax.set_xlabel('Customer ID / Kunden-ID')
ax.set_ylabel('Total Spent / Gesamtausgaben (£)')
plt.tight_layout()
plt.show()

print('\n🇬🇧 Insight: Top customers are likely B2B wholesale buyers.')
print('   Build dedicated account management relationships with them.')
print('\n🇩🇪 Erkenntnis: Top-Kunden sind wahrscheinlich B2B-Großhandelskäufer.')
print('   Dedizierte Key-Account-Beziehungen aufbauen.')

## Query 6 — RFM Segmentation / RFM-Segmentierung

In [ ]:
# EN Full RFM scoring with segment labels
# DE Vollständige RFM-Bewertung mit Segmentbezeichnungen
rfm_sql = pd.read_sql("""
WITH rfm_raw AS (
    SELECT
        CustomerID,
        CAST(JULIANDAY('2011-12-10') - JULIANDAY(MAX(InvoiceDate)) AS INTEGER) AS recency_days,
        COUNT(DISTINCT InvoiceNo)                                               AS frequency,
        ROUND(SUM(Revenue), 2)                                                  AS monetary
    FROM   clean_sales
    GROUP  BY CustomerID
),
rfm_scored AS (
    SELECT *,
        CASE
            WHEN recency_days <= 30  THEN 4
            WHEN recency_days <= 90  THEN 3
            WHEN recency_days <= 180 THEN 2
            ELSE 1
        END AS r_score,
        CASE
            WHEN frequency >= 10 THEN 4
            WHEN frequency >= 5  THEN 3
            WHEN frequency >= 2  THEN 2
            ELSE 1
        END AS f_score,
        CASE
            WHEN monetary >= 5000 THEN 4
            WHEN monetary >= 1000 THEN 3
            WHEN monetary >= 300  THEN 2
            ELSE 1
        END AS m_score
    FROM rfm_raw
),
segmented AS (
    SELECT *,
        CASE
            WHEN r_score = 4 AND f_score = 4   THEN 'Champions'
            WHEN r_score >= 3 AND f_score >= 3  THEN 'Loyal Customers'
            WHEN r_score = 4                   THEN 'Recent Customers'
            WHEN r_score = 3                   THEN 'Potential Loyalists'
            WHEN r_score = 2 AND f_score >= 2  THEN 'At Risk'
            WHEN r_score <= 2 AND f_score <= 2 THEN 'Lost'
            ELSE 'Need Attention'
        END AS segment
    FROM rfm_scored
)
SELECT
    segment,
    COUNT(*)                     AS customers,
    ROUND(AVG(recency_days), 0)  AS avg_recency,
    ROUND(AVG(frequency), 1)     AS avg_frequency,
    ROUND(AVG(monetary), 0)      AS avg_monetary,
    ROUND(SUM(monetary), 0)      AS total_revenue
FROM   segmented
GROUP  BY segment
ORDER  BY total_revenue DESC
""", conn)

display(rfm_sql)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
palette = sns.color_palette('Set2', len(rfm_sql))

axes[0].pie(rfm_sql['customers'], labels=rfm_sql['segment'],
            autopct='%1.1f%%', colors=palette, startangle=140)
axes[0].set_title('Customer Segments\nKundensegmente', fontweight='bold')

axes[1].barh(rfm_sql['segment'][::-1], rfm_sql['total_revenue'][::-1], color=palette[::-1])
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'£{x/1000:.0f}k'))
axes[1].set_title('Revenue by Segment / Umsatz nach Segment', fontweight='bold')

plt.tight_layout()
plt.show()

print('\n🇬🇧 Insight:')
print('   Champions      -> protect with VIP loyalty programme')
print('   At Risk        -> win-back campaign with personalised discount')
print('   Lost           -> deprioritise, re-acquisition cost exceeds LTV')
print('\n🇩🇪 Erkenntnis:')
print('   Champions      -> mit VIP-Treueprogramm schützen')
print('   At Risk        -> Rückgewinnungskampagne mit persönlichem Rabatt')
print('   Lost           -> deprioritisieren, Wiedergewinnungskosten übersteigen LTV')

## Query 7 — Cohort Retention / Kohortenretention

In [ ]:
# EN Retention rate per acquisition cohort
# DE Bindungsrate pro Akquisitionskohorte
cohort_sql = pd.read_sql("""
WITH first_purchase AS (
    SELECT CustomerID,
           STRFTIME('%Y-%m', MIN(InvoiceDate)) AS cohort_month
    FROM   clean_sales
    GROUP  BY CustomerID
),
activity AS (
    SELECT s.CustomerID,
           STRFTIME('%Y-%m', s.InvoiceDate) AS activity_month,
           f.cohort_month
    FROM   clean_sales s
    JOIN   first_purchase f ON s.CustomerID = f.CustomerID
),
cohort_size AS (
    SELECT cohort_month, COUNT(DISTINCT CustomerID) AS cohort_customers
    FROM   first_purchase
    GROUP  BY cohort_month
)
SELECT
    a.cohort_month,
    cs.cohort_customers,
    COUNT(DISTINCT CASE WHEN a.activity_month > a.cohort_month THEN a.CustomerID END) AS returned,
    ROUND(
        COUNT(DISTINCT CASE WHEN a.activity_month > a.cohort_month THEN a.CustomerID END)
        * 100.0 / cs.cohort_customers, 1
    ) AS retention_pct
FROM      activity a
JOIN      cohort_size cs ON a.cohort_month = cs.cohort_month
GROUP BY  a.cohort_month, cs.cohort_customers
ORDER BY  a.cohort_month
""", conn)

display(cohort_sql)

fig, ax = plt.subplots(figsize=(10, 5))
avg_ret = cohort_sql['retention_pct'].mean()
ax.bar(cohort_sql['cohort_month'], cohort_sql['retention_pct'],
       color='steelblue', alpha=0.85)
ax.axhline(avg_ret, color='red', linestyle='--',
           label=f'Avg / Durchschn. {avg_ret:.1f}%')
ax.set_title('Retention Rate by Cohort / Bindungsrate nach Kohorte', fontweight='bold')
ax.set_ylabel('Retention % / Bindungsrate %')
ax.set_xlabel('Cohort Month / Kohortenmonat')
ax.tick_params(axis='x', rotation=45)
ax.legend()
plt.tight_layout()
plt.show()

print(f'\n🇬🇧 Insight: Avg retention across all cohorts: {avg_ret:.1f}%.')
print('   A 30-day post-purchase email sequence is the highest-ROI action to improve this metric.')
print(f'\n🇩🇪 Erkenntnis: Durchschnittliche Bindungsrate über alle Kohorten: {avg_ret:.1f}%.')
print('   Eine 30-Tage-E-Mail-Sequenz nach dem Kauf ist die Maßnahme mit dem höchsten ROI.')

## Query 8 — Day x Hour Revenue Heatmap / Umsatz-Heatmap Tag x Stunde

In [ ]:
# EN Revenue breakdown by day of week and hour
# DE Umsatzaufschlüsselung nach Wochentag und Stunde
heatmap_sql = pd.read_sql("""
SELECT
    CASE CAST(STRFTIME('%w', InvoiceDate) AS INTEGER)
        WHEN 1 THEN 'Mon' WHEN 2 THEN 'Tue' WHEN 3 THEN 'Wed'
        WHEN 4 THEN 'Thu' WHEN 5 THEN 'Fri' WHEN 6 THEN 'Sat'
    END                                          AS day_of_week,
    CAST(STRFTIME('%H', InvoiceDate) AS INTEGER) AS hour_of_day,
    ROUND(SUM(Revenue), 0)                       AS revenue
FROM   clean_sales
WHERE  CAST(STRFTIME('%w', InvoiceDate) AS INTEGER) BETWEEN 1 AND 5
GROUP  BY day_of_week, hour_of_day
ORDER  BY day_of_week, hour_of_day
""", conn)

pivot     = heatmap_sql.pivot(index='day_of_week', columns='hour_of_day', values='revenue')
day_order = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri']
pivot     = pivot.reindex(day_order)

fig, ax = plt.subplots(figsize=(14, 4))
sns.heatmap(pivot, ax=ax, cmap='YlOrRd', linewidths=0.3,
            cbar_kws={'label': 'Revenue / Umsatz (£)'})
ax.set_title('Revenue Heatmap: Day x Hour / Umsatz-Heatmap: Tag x Stunde', fontweight='bold')
ax.set_xlabel('Hour of Day / Stunde')
ax.set_ylabel('Day of Week / Wochentag')
plt.tight_layout()
plt.show()

print('\n🇬🇧 Insight: Peak buying window is Tue-Thu, 9:00-12:00.')
print('   Schedule marketing emails and promotions during these hours.')
print('\n🇩🇪 Erkenntnis: Spitzenkaufzeit ist Di-Do, 9:00-12:00.')
print('   Marketing-E-Mails und Aktionen während dieser Zeiten planen.')

## Query 9 — Product Return Rate / Produktrückgabequote

In [ ]:
# EN Products with highest return rates — quality signal
# DE Produkte mit höchster Rückgabequote — Qualitätssignal
returns = pd.read_sql("""
WITH sold AS (
    SELECT Description, SUM(Quantity) AS qty_sold
    FROM   sales
    WHERE  Quantity > 0 AND UnitPrice > 0
    GROUP  BY Description
),
returned AS (
    SELECT Description, ABS(SUM(Quantity)) AS qty_returned
    FROM   sales
    WHERE  Quantity < 0
    GROUP  BY Description
)
SELECT
    s.Description,
    s.qty_sold,
    COALESCE(r.qty_returned, 0)                                 AS qty_returned,
    ROUND(COALESCE(r.qty_returned, 0) * 100.0 / s.qty_sold, 1) AS return_rate_pct
FROM      sold s
LEFT JOIN returned r ON s.Description = r.Description
WHERE     s.qty_sold >= 50
ORDER BY  return_rate_pct DESC
LIMIT     15
""", conn)

display(returns)

fig, ax = plt.subplots(figsize=(12, 5))
ax.barh(returns['Description'][::-1], returns['return_rate_pct'][::-1],
        color='salmon', alpha=0.85)
ax.set_title('Top 15 Products by Return Rate / Top-15-Produkte nach Rückgabequote',
             fontweight='bold')
ax.set_xlabel('Return Rate % / Rückgabequote %')
plt.tight_layout()
plt.show()

print('\n🇬🇧 Insight: High return rates signal quality issues or misleading product descriptions.')
print('   Investigate top offenders — fix product pages or review supplier quality.')
print('\n🇩🇪 Erkenntnis: Hohe Rückgabequoten signalisieren Qualitätsprobleme oder irreführende Produktbeschreibungen.')
print('   Spitzenreiter untersuchen — Produktseiten verbessern oder Lieferantenqualität prüfen.')

## Query 10 — Geographic Revenue Breakdown / Geografische Umsatzverteilung

In [ ]:
# EN Revenue share by country with AOV
# DE Umsatzanteil nach Land mit Ø-Bestellwert
geo = pd.read_sql("""
WITH country_stats AS (
    SELECT
        Country,
        ROUND(SUM(Revenue), 0)             AS revenue,
        COUNT(DISTINCT CustomerID)         AS customers,
        COUNT(DISTINCT InvoiceNo)          AS orders,
        ROUND(SUM(Revenue)
            / COUNT(DISTINCT InvoiceNo),2) AS aov
    FROM   clean_sales
    GROUP  BY Country
),
total AS (SELECT SUM(revenue) AS grand_total FROM country_stats)
SELECT
    c.Country, c.revenue, c.customers, c.orders, c.aov,
    ROUND(c.revenue * 100.0 / t.grand_total, 2) AS revenue_share_pct
FROM   country_stats c, total t
ORDER  BY c.revenue DESC
LIMIT  15
""", conn)

display(geo)

geo_intl = geo[geo['Country'] != 'United Kingdom'].head(10)
fig, ax  = plt.subplots(figsize=(10, 5))
ax.barh(geo_intl['Country'][::-1], geo_intl['revenue'][::-1],
        color='teal', alpha=0.8)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'£{x/1000:.0f}k'))
ax.set_title('Top 10 International Markets / Top-10 internationale Märkte', fontweight='bold')
ax.set_xlabel('Revenue / Umsatz (£)')
plt.tight_layout()
plt.show()

uk_share = geo.loc[geo['Country'] == 'United Kingdom', 'revenue_share_pct'].values
if len(uk_share) > 0:
    print(f'\n🇬🇧 Insight: UK accounts for {uk_share[0]:.1f}% of revenue.')
print('   Netherlands, EIRE, Germany are high-potential international markets.')
if len(uk_share) > 0:
    print(f'\n🇩🇪 Erkenntnis: UK macht {uk_share[0]:.1f}% des Umsatzes aus.')
print('   Niederlande, Irland und Deutschland sind internationale Wachstumsmärkte.')

conn.close()
print('\n✅ SQL analysis complete / SQL-Analyse abgeschlossen.')